# 🐋 Blue Whale — Colab GPU Server

**CNN Ensemble chạy trên GPU Colab cho nhanh (< 1s/ảnh)**

Máy local chỉ lo phần còn lại (OOD, abstain, Hội Đoàn, chat, TTS, UI).  
Nếu Colab tắt giữa chừng → local tự fallback CPU, KHÔNG crash.

---
### Hướng dẫn:
1. Runtime → Change runtime type → **GPU (T4)** → Save
2. Chạy từng Cell theo thứ tự 1 → 2 → 3 → 4
3. Chờ dòng `[COLAB] GPU đã warm up — sẵn sàng!`
4. URL tunnel sẽ tự push về app — hoặc copy paste thủ công
5. Trên máy local: badge chuyển "⚡ GPU Colab" → bắt đầu demo

## CELL 1 — Cài thư viện

In [1]:
!pip install -q fastapi uvicorn nest_asyncio pillow opencv-python-headless requests
# !pip install -q pyngrok   # bật dòng này nếu dùng ngrok fallback (Cell 4 fallback)

## CELL 2 — Tải model từ GitHub Release
Không cần Google Drive, không cần upload, không cần đăng nhập.

In [2]:
import os, time, urllib.request

MODEL_DIR = '/content/models'
os.makedirs(MODEL_DIR, exist_ok=True)

RELEASE = "https://github.com/levan1410/BlueWhale_models/releases/download/v1.0.0"
FILES = [
    "model_0_EfficientNetV2-S.keras",
    "model_1_ResNet50V2.keras",
    "model_2_ConvNeXt-Tiny.keras",
    "class_names.json",
    "ensemble_config.json",
]

print("Tải model từ GitHub Release (~750MB, Colab tải rất nhanh)...")
t0 = time.time()
for f in FILES:
    dst = os.path.join(MODEL_DIR, f)
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        print(f"  - {f}  (đã có, bỏ qua)")
        continue
    print(f"  ↓ {f} ...", end=" ", flush=True)
    try:
        urllib.request.urlretrieve(f"{RELEASE}/{f}", dst)
        print(f"{os.path.getsize(dst)/1024/1024:.0f} MB")
    except Exception as e:
        print(f"LỖI: {e}")
        print(f"    → Kiểm tra release tồn tại: {RELEASE.replace('/download/', '/tag/')}")
        raise

print(f"\nXong ({time.time()-t0:.0f}s) — chạy tiếp Cell 3 & 4.")

Tải model từ GitHub Release (~750MB, Colab tải rất nhanh)...
  ↓ model_0_EfficientNetV2-S.keras ... 238 MB
  ↓ model_1_ResNet50V2.keras ... 223 MB
  ↓ model_2_ConvNeXt-Tiny.keras ... 248 MB
  ↓ class_names.json ... 0 MB
  ↓ ensemble_config.json ... 0 MB

Xong (27s) — chạy tiếp Cell 3 & 4.


### CELL 2 (DỰ PHÒNG) — Lấy từ Google Drive nếu không có mạng GitHub

In [3]:
# BỎ COMMENT CELL NÀY NẾU GITHUB RELEASE KHÔNG TẢI ĐƯỢC

# from google.colab import drive
# import os, shutil, glob
#
# MODEL_DIR = '/content/models'
# os.makedirs(MODEL_DIR, exist_ok=True)
# drive.mount('/content/drive', force_remount=False)
#
# keras_found = (glob.glob('/content/drive/MyDrive/*/*/*.keras')
#              + glob.glob('/content/drive/MyDrive/*/*.keras')
#              + glob.glob('/content/drive/MyDrive/*.keras'))
# if not keras_found:
#     raise FileNotFoundError("Không thấy .keras trên Drive — upload 5 file rồi chạy lại")
#
# DRIVE_DIR = os.path.dirname(keras_found[0])
# on_drive  = os.listdir(DRIVE_DIR)
# for f in [x for x in on_drive if x.endswith('.keras')] + \
#          [x for x in ('class_names.json', 'ensemble_config.json') if x in on_drive]:
#     src, dst = os.path.join(DRIVE_DIR, f), os.path.join(MODEL_DIR, f)
#     if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src)):
#         shutil.copy2(src, dst)
#     print(f"  ✓ {f}")
# print("Xong — chạy tiếp Cell 3 & 4.")

## CELL 3 — Server code (CNN Ensemble + FastAPI)

In [4]:
import os, io, json, glob, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import numpy as np
import cv2
from PIL import Image
import tensorflow as tf
import keras
from tensorflow.keras.models import load_model
from fastapi import FastAPI, File, UploadFile, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response as _FastResponse
import requests as _rq

MODEL_DIR = os.environ.get("MODEL_DIR", "/content/models")

# Cấp phát VRAM tăng dần
for _gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except Exception:
        pass


@keras.saving.register_keras_serializable()
class FocalLoss(keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=1.0, label_smoothing=0.1,
                 name="focal_loss", reduction="sum_over_batch_size", **kwargs):
        super().__init__(name=name, reduction=reduction)
        self.gamma = gamma; self.alpha = alpha; self.label_smoothing = label_smoothing
    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        y_pred = tf.nn.softmax(y_pred, axis=-1)
        if self.label_smoothing > 0:
            n = tf.cast(tf.shape(y_true)[-1], tf.float32)
            y_true = y_true * (1 - self.label_smoothing) + self.label_smoothing / n
        ce = -y_true * tf.math.log(y_pred)
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        return tf.reduce_mean(tf.reduce_sum(self.alpha * tf.pow(1.0 - p_t, self.gamma) * ce, axis=-1))
    def get_config(self):
        c = super().get_config()
        c.update({"gamma": self.gamma, "alpha": self.alpha, "label_smoothing": self.label_smoothing})
        return c


# ── Load ensemble models ──
STATE = {}

def _load_all():
    print(f"  MODEL_DIR = {MODEL_DIR}")
    if not os.path.isdir(MODEL_DIR):
        print(f"  [LỖI] Thư mục {MODEL_DIR} KHÔNG TỒN TẠI.")
        STATE.update(models=[], names=[], weights=[], input_size=224, classes=[])
        return
    all_files = os.listdir(MODEL_DIR)
    print(f"  Có {len(all_files)} file: {all_files[:8]}")

    keras_paths = sorted(glob.glob(os.path.join(MODEL_DIR, "model_*.keras")))
    if not keras_paths:
        print(f"  [LỖI] Không tìm thấy file 'model_*.keras'")

    models, names = [], []
    for p in keras_paths:
        try:
            m = load_model(p, custom_objects={"FocalLoss": FocalLoss})
            models.append(m); names.append(os.path.basename(p))
            print(f"  ✓ {os.path.basename(p)}")
        except Exception as e:
            print(f"  [LỖI] {os.path.basename(p)}: {e}")

    cfg_path = os.path.join(MODEL_DIR, "ensemble_config.json")
    try:
        cfg = json.load(open(cfg_path))
        weights = cfg.get("weights", [1.0] * len(models))[:len(models)]
        input_size = cfg.get("input_size", 224)
    except Exception:
        weights, input_size = [1.0] * len(models), 224

    cls_path = os.path.join(MODEL_DIR, "class_names.json")
    try:
        d = json.load(open(cls_path))
        classes = d if isinstance(d, list) else list(d.keys())
    except Exception:
        classes = []

    STATE.update(models=models, names=names, weights=weights,
                 input_size=input_size, classes=classes)
    print(f"  Ensemble: {len(models)} models, size={input_size}, classes={len(classes)}")

_load_all()


def predict_image(pil_img):
    models  = STATE["models"]; weights = STATE["weights"]
    sz      = STATE["input_size"]; classes = STATE["classes"]
    img = cv2.resize(np.array(pil_img.convert("RGB")), (sz, sz))
    inp = np.stack([img, img[:, ::-1, :]]).astype(np.float32)
    tw  = sum(weights[:len(models)]) or 1
    combined = None
    per_model = []
    for mdl, w in zip(models, weights):
        out = mdl.predict(inp, verbose=0)
        sm  = tf.nn.softmax(out, axis=-1).numpy().mean(axis=0)
        per_model.append(sm)
        p = sm * (w / tw)
        if combined is None:
            combined = np.zeros_like(p)
        n = min(len(combined), len(p))
        combined[:n] += p[:n]
    n_out = len(combined)
    cls = list(classes)
    if len(cls) < n_out:
        cls = cls + [f"class_{i}" for i in range(len(cls), n_out)]
    elif len(cls) > n_out:
        cls = cls[:n_out]
    idx  = int(np.argmax(combined))
    conf = float(combined[idx])
    order = np.argsort(-combined)[:5]
    top5 = [{"label": cls[i], "conf": round(float(combined[i]) * 100, 1)} for i in order]
    all_probs = {cls[i]: round(float(combined[i]), 4) for i in range(len(cls))}
    # Disagreement
    disagreement = None
    if len(per_model) >= 2:
        try:
            names_l = STATE.get("names", [])
            def _nice(nm, i):
                b = str(nm).rsplit('/', 1)[-1].replace('.keras', '')
                for pre in ('model_0_', 'model_1_', 'model_2_', 'model_'):
                    if b.startswith(pre): b = b[len(pre):]; break
                return b or f"Model {i+1}"
            votes = []
            for i, sm in enumerate(per_model):
                vi = int(np.argmax(sm))
                votes.append({"model": _nice(names_l[i] if i < len(names_l) else "", i),
                              "label": cls[vi] if vi < len(cls) else f"class_{vi}",
                              "conf": round(float(sm[vi]) * 100, 1),
                              "agrees": bool(vi == idx)})
            top1_each = [int(np.argmax(sm)) for sm in per_model]
            n_agree = sum(1 for t in top1_each if t == idx)
            disagreement = {
                "n_models": len(per_model),
                "agree_top1": n_agree,
                "agree_ratio": round(n_agree / len(per_model), 3),
                "std_top": round(float(np.std([float(sm[idx]) for sm in per_model])), 4),
                "votes": votes,
            }
        except Exception:
            disagreement = None
    return cls[idx], conf, top5, all_probs, disagreement


# ── FastAPI App ──
app = FastAPI(title="Blue Whale Colab GPU Server")
app.add_middleware(CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


@app.get("/health")
def health():
    return {
        "ok": len(STATE.get("models", [])) > 0,
        "models": len(STATE.get("models", [])),
        "classes": len(STATE.get("classes", [])),
        "gpu": len(tf.config.list_physical_devices("GPU")) > 0,
    }


@app.get("/warmup")
def warmup():
    dummy = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
    label, conf, top5, _, _ = predict_image(dummy)
    return {"status": "warmed_up",
            "gpu": len(tf.config.list_physical_devices("GPU")) > 0,
            "test_label": label}


@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    if not STATE.get("models"):
        raise HTTPException(503, "Models not loaded")
    try:
        img_bytes = await file.read()
        pil_img = Image.open(io.BytesIO(img_bytes))
        label, conf, top5, all_probs, disagreement = predict_image(pil_img)
        return {
            "status": "ok",
            "label": label,
            "confidence": round(conf * 100, 1),
            "top5": top5,
            "all_probs": all_probs,
            "disagreement": disagreement,
        }
    except Exception as e:
        raise HTTPException(500, str(e))


# ── Proxy Ollama (nếu có chạy Cell 3.5) ──
_OLLAMA_LOCAL = "http://localhost:11434"

@app.get("/ollama/api/tags")
def ollama_tags_proxy():
    try:
        r = _rq.get(f"{_OLLAMA_LOCAL}/api/tags", timeout=5)
        return _FastResponse(content=r.content, status_code=r.status_code,
                             media_type="application/json")
    except Exception as e:
        raise HTTPException(503, f"Ollama chưa chạy trên Colab: {e}")

@app.post("/ollama/api/chat")
async def ollama_chat_proxy(request: Request):
    try:
        body = await request.body()
        r = _rq.post(f"{_OLLAMA_LOCAL}/api/chat", data=body,
                     headers={"Content-Type": "application/json"}, timeout=180)
        return _FastResponse(content=r.content, status_code=r.status_code,
                             media_type="application/json")
    except Exception as e:
        raise HTTPException(503, f"Ollama lỗi: {e}")

@app.post("/ollama/api/generate")
async def ollama_generate_proxy(request: Request):
    try:
        body = await request.body()
        r = _rq.post(f"{_OLLAMA_LOCAL}/api/generate", data=body,
                     headers={"Content-Type": "application/json"}, timeout=180)
        return _FastResponse(content=r.content, status_code=r.status_code,
                             media_type="application/json")
    except Exception as e:
        raise HTTPException(503, f"Ollama lỗi: {e}")


print("\n✅ Server code loaded — chạy Cell 3.5 (tuỳ chọn) rồi Cell 4 để khởi động.")

  MODEL_DIR = /content/models
  Có 5 file: ['class_names.json', 'ensemble_config.json', 'model_2_ConvNeXt-Tiny.keras', 'model_0_EfficientNetV2-S.keras', 'model_1_ResNet50V2.keras']
  ✓ model_0_EfficientNetV2-S.keras
  ✓ model_1_ResNet50V2.keras
  ✓ model_2_ConvNeXt-Tiny.keras
  Ensemble: 3 models, size=224, classes=38

✅ Server code loaded — chạy Cell 3.5 (tuỳ chọn) rồi Cell 4 để khởi động.


## CELL 3.5 (TUỲ CHỌN) — Cài Ollama trên Colab GPU

**Lợi ích:**
- Gemma chạy trên GPU T4 → trả lời trong 1-3 giây (CPU local: 30-120s)
- Hội Đoàn SINH CÂU HỎI THÔNG MINH động
- Cross-val + chat + kết luận Hội Đoàn đều nhanh
- Máy local KHÔNG tốn RAM cho Gemma nữa

In [9]:
# CELL 3.5 — Cài Ollama trên Colab GPU
# QUAN TRỌNG: Dùng ! shell commands (Colab chặn os.system cho một số URL)

import subprocess, time, requests

print("[1/3] Cài Ollama...")


[1/3] Cài Ollama...


In [10]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


In [14]:
!apt-get install -y zstd


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (2,752 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [15]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [16]:
import subprocess, time, os, requests

ollama_path = "/usr/local/bin/ollama"

# Khởi động server ngầm (systemd không chạy trên Colab nên phải tự start)
print("[2/3] Khởi động Ollama server...")
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
subprocess.Popen([ollama_path, "serve"], env=env,
                 stdout=open("/tmp/ollama.log", "w"),
                 stderr=subprocess.STDOUT)

for i in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            print("      ✓ Ollama server OK")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("      [!] Server chưa lên — xem log:")
    os.system("cat /tmp/ollama.log")


[2/3] Khởi động Ollama server...
      ✓ Ollama server OK


In [17]:
# Tải model Gemma (chạy trên GPU T4)
!ollama pull gemma4:12b || ollama pull gemma4:e2b || ollama pull gemma3:4b-it-qat


In [18]:
# Xác nhận
import requests
r = requests.get("http://localhost:11434/api/tags", timeout=5)
models = [m["name"] for m in r.json().get("models", [])]
print(f"✅ Ollama sẵn sàng: {models}")
!nvidia-smi


✅ Ollama sẵn sàng: ['gemma4:12b']
Fri Jun 12 08:14:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P0             28W /   70W |    1129MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------

## CELL 4 — Khởi động + Cloudflared tunnel (KHÔNG cần đăng ký)

- Miễn phí 100%, KHÔNG cần đăng ký, KHÔNG cần token
- URL tự động push về app qua ntfy.sh relay
- `CHAY_APP.bat` sẽ tự lấy URL và kết nối

In [ ]:
import os, re, time, threading, subprocess, requests
import nest_asyncio, uvicorn

nest_asyncio.apply()

# 1. Cài cloudflared
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("[1/4] Cài cloudflared...")
    os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")
    print("      Xong.")
else:
    print("[1/4] cloudflared đã cài sẵn.")

# 2. Mở tunnel
print("[2/4] Mở tunnel cloudflared...")
os.system("pkill -f cloudflared 2>/dev/null")
time.sleep(1)

_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000",
     "--no-autoupdate", "--metrics", "localhost:0"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

PUBLIC_URL = None
_deadline = time.time() + 30
for line in iter(_proc.stdout.readline, ""):
    if time.time() > _deadline:
        break
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0)
        break

if not PUBLIC_URL:
    raise RuntimeError("Không lấy được URL cloudflared sau 30 giây. Thử chạy lại cell.")

print("=" * 52)
print(f"🌐 URL CLOUDFLARED: {PUBLIC_URL}")
print("=" * 52)
print(f"Health: {PUBLIC_URL}/health")
print(f"Warmup: {PUBLIC_URL}/warmup")

# 3. Push URL lên ntfy.sh relay
_RELAY_TOPIC = "bluewhale-colab-ccct2025-x7k9m"
print("[3/4] Push URL lên relay...")
try:
    _r = requests.post(f"https://ntfy.sh/{_RELAY_TOPIC}", data=PUBLIC_URL.encode(), timeout=6)
    if _r.status_code == 200:
        print(f"      ✓ Đã push — CHAY_APP.bat sẽ tự kết nối GPU!")
    else:
        print(f"      [WARN] Push thất bại ({_r.status_code})")
except Exception as _e:
    print(f"      [WARN] Lỗi push: {_e}")

# 4. Warmup GPU + chạy server
print("[4/4] Warmup GPU (~10-20s lần đầu)...")
def auto_warmup():
    time.sleep(3)
    try:
        requests.get("http://localhost:8000/warmup", timeout=30)
        print("\n🐋 [COLAB] GPU đã warm up — sẵn sàng!")
    except Exception as e:
        print(f"[COLAB] Warmup lỗi (vẫn chạy được): {e}")
threading.Thread(target=auto_warmup, daemon=True).start()

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
await server.serve()

[1/4] Cài cloudflared...
      Xong.
[2/4] Mở tunnel cloudflared...
🌐 URL CLOUDFLARED: https://note-lenders-titled-emissions.trycloudflare.com
Health: https://note-lenders-titled-emissions.trycloudflare.com/health
Warmup: https://note-lenders-titled-emissions.trycloudflare.com/warmup
[3/4] Push URL lên relay...
      ✓ Đã push — CHAY_APP.bat sẽ tự kết nối GPU!
[4/4] Warmup GPU (~10-20s lần đầu)...
[COLAB] Warmup lỗi (vẫn chạy được): HTTPConnectionPool(host='localhost', port=8000): Read timed out. (read timeout=30)


## CELL 4 (FALLBACK) — Dùng NGROK nếu bạn CÓ authtoken

Chỉ chạy cell này **THAY CHO** Cell 4 ở trên nếu cloudflared không hoạt động.

In [ ]:
# NGROK_TOKEN = "PASTE_TOKEN_CUA_BAN_VAO_DAY"
#
# import nest_asyncio, threading, time, requests, uvicorn
# from pyngrok import ngrok, conf
#
# nest_asyncio.apply()
# ngrok.kill()
# conf.get_default().auth_token = NGROK_TOKEN
#
# tunnel = ngrok.connect(8000)
# PUBLIC_URL = tunnel.public_url
#
# print("=" * 52)
# print(f"🌐 URL NGROK: {PUBLIC_URL}")
# print("=" * 52)
#
# _RELAY_TOPIC = "bluewhale-colab-ccct2025-x7k9m"
# try:
#     requests.post(f"https://ntfy.sh/{_RELAY_TOPIC}", data=PUBLIC_URL.encode(), timeout=6)
#     print("[RELAY] ✓ Đã push URL")
# except Exception as e:
#     print(f"[RELAY] Lỗi: {e}")
#
# def auto_warmup():
#     time.sleep(3)
#     try:
#         requests.get("http://localhost:8000/warmup", timeout=30)
#         print("\n🐋 [COLAB] GPU đã warm up — sẵn sàng!")
#     except Exception as e:
#         print(f"[COLAB] Warmup lỗi: {e}")
# threading.Thread(target=auto_warmup, daemon=True).start()
#
# uvicorn.run(app, host="0.0.0.0", port=8000)